In [1]:
import os

import rasterio
import yaml

from pathlib import Path

In [2]:
os.chdir("..")

In [3]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [4]:
# project paths
data_dir = Path(config["data_dir"])
data_dir_raw = data_dir / "raw"
aoi_path = data_dir / "aoi.geojson"

bands = [path for path in data_dir.rglob("*.jp2")]
bands_dir = list(set([band.parent for band in bands]))[0]

In [17]:
scale_factor = 10000

### Helper functions

In [5]:
# load bands
def load_band(path):
    with rasterio.open(path, "r") as src:
        return {
            "data": src.read(1),
            "metadata": {
                "band_path": path,
                "profile": src.profile,
                "crs": src.crs,
                "transform": src.transform,
                "res": src.res,
                "nodata": src.nodata
            }
        }

In [20]:
# scale digital numbers to surface reflectance
def scale_reflectance(band, scale_factor):
    return band / scale_factor

### Data preparation

In [18]:
bands = {}
for band in data_dir_raw.rglob("*.jp2"):
    b = band.name[-11:-8]
    bands[b] = load_band(band)
    bands[b]["data"] = scale_reflectance(bands[b]["data"], scale_factor)